# Generazione di un dataset avanzato usando Gemma 4

# Descrizione del problema

All'interno di questo Jupyter Notebook è presente il contenuto del punto 6 del [Notebook 1: Generazione Dataset Elementare](01_Generazione_Dataset_Elementare.ipynb)

Per la generazione di un dataset avanzato ho deciso di chiedere l'aiuto delle LLM. Invece di optare per una normale query all'interno di interfacce web commerciali (come Gemini o soluzioni simili), ho deciso di utilizzare uno strumento molto più versatile e professionale, ovvero Gemma 4. Questo è un modello linguistico di grandi dimensione (Large Language Model) sviluppato da Google che ho configurato ed eseguito localmente tramite la piattaforma Ollama. Questa architettura ha permesso di aggirare i vincoli legati al rate limiting e i costi di consumo tipici delle API basate su cloud (come ad esempio Gemini, anch'essa di Google), garantendo al contempo il controllo totale sui parametri di generazione (come ad esempio la formattazione nativa della risposta). Per maggiori informazioni, è possibile consultare la documentazione ufficiale di [Gemma 4](https://ai.google.dev/gemma/docs/core/model_card_4?hl=it).

In questo dataset ho deciso appositamente di ignorare la creazione di finti nomi e finte email in quanto non richieste dal PW.

### Come eseguire il codice di questo Jupyter Notebook

__Questo Jupyter Notebook contiene già del codice eseguito a scopo dimostrativo, tuttavia a seguire è disponibile una breve guida passo-passo qualora si desiderasse configurare l'ambiente ed eseguirlo nuovamente__

Per eseguire correttamente questo Notebook e avviare la generazione del dataset tramite il modello locale, segui questi passaggi preliminari:

- Ollama deve essere installato direttamente sul tuo sistema operativo per poter ospitare il modello in locale.
* **Windows / Mac / Linux:** Scarica e installa l'applicazione ufficiale dal sito [ollama.com](https://ollama.com).
- Dopo di che, va scaricato ed installato il modello. Da powershell, basta runnare il comando: *ollama run gemma4* (9.6 GB richiesti)
- Una volta installato, per rimuovere il modello basta utilizzare il comando: *ollama rm gemma4*

In [78]:
%pip install pandas ollama pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Sviluppo del codice

Import delle librerie:

In [79]:
import json
import pandas as pd
import random
from ollama import Client
from pydantic import BaseModel, Field

Struttura base dei parametri di generazione:

In [80]:
class TicketSintetico(BaseModel):
    title: str = Field(
        description="Un titolo breve, specifico e realistico."
    )
    body: str = Field(
        description=(
            "Il corpo del ticket. Deve contenere una descrizione breve (1-2 frasi) del problema reale. "
            "NON usare placeholder, parentesi quadre, simboli di markup o istruzioni di compilazione (es. '[Descrivere qui...]'). "
            "Non è obbligatorio inserire nominativi di applicazioni/servizi ma, se lo fai, utilizza nomi verosimili (non usare ad esempio modulo X o endpoint x)"
            "Il testo deve essere testo pronto, discorsivo e già compilato, scritto dal punto di vista dell'utente o del dipendente che sta riscontrando il problema in prima persona"
        )
    )

Funzione della generazione ticket con gemma. Qui viene formattato il prompt della richiesta da fare a Gemma per la creazione di un determinato ticket. I parametri sono:
- categoria: la categoria del ticket
- priorità: la priorità del problema
- esperienza: il livello di esperienza nel settore specifico della categoria da parte dell'utente
- colloqualità: il tono da utilizzare per la stesura del ticket
- nome_utente: informazione extra da utilizzare in maniera facoltativa che rappresenta se voler aggiungere il nome dell'utente durante la stesura del ticket
- typo: booleano che specifica se richiedere l'inserimento di un typo o meno

Il prompt finale nasce da diverse prove al fine di ottenere un risultato più realistico possibile

L'output della funzione sarà una tupla, contenente body e il title del ticket.

In [ ]:
client = Client()
def genera_ticket_con_gemma(categoria, priorita,esperienza,colloqualità,incipit,titolo,disturbo=False,typo=False):
    prompt_utente = f"""Genera un ticket di assistenza aziendale in lingua italiana con queste caratteristiche:
    - Categoria del ticket: {categoria}. Questo rappresenta l'argomento principale di cui deve parlare il ticket.
    - Priorità del problema: {priorita}.
    - Livello di esperienza dell'utente: {esperienza}.
    - Livello di colloquialità/tono: {colloqualità}
  
    STILE DI SCRITTURA:
    1. Sii estremamente sintetico. Il testo deve essere composto da massimo 2-3 frasi corte.
    2. Scrivi in prima persona. 
    3. Non usare MAI caratteri speciali di formattazione come asterischi (*), corsivi o grassetti.
    4. Evita deliri o flussi di coscienza lunghi: vai dritto al problema reale.
    5. La forma di saluto iniziale deve essere esattamente "{incipit}" (se non è vuota) o una sua variante naturale.
    6. Usa termini tecnici, acronimi o parole chiave specifiche in base alla categoria del ticket ({categoria}) e all'esperienza ({esperienza}).
    7. Il titolo deve simulare l'oggetto scritto da un utente reale (breve, sbrigativo o imperfetto). Inizia il titolo UTILIZZANDO OBBLIGATORIAMENTE questa esatta espressione iniziale: "{titolo}". Continua poi la frase completando l'oggetto in modo che si colleghi perfettamente alla categoria ({categoria}) e soprattutto alla priorità ({priorita}).
    8. Rispetta rigorosamente la coerenza: se la priorità è Bassa, non inserire scadenze imminenti o blocchi operativi nel titolo o nel corpo
    """
    if disturbo:
        prompt_utente+="""\n - VINCOLO DI CONTESTO DINAMICO (OBBLIGATORIO):
                        - {disturbo}. Integra questo dettaglio nel testo in modo naturale, usandolo come elemento di sfondo, senza farlo diventare il problema principale."""

    if typo:
        prompt_utente += "\n- Inserisci dove rendi opportuno un typo per rendere il ticket più human like."

    try:
        response = client.chat(
            model='gemma4:e2b', 
            messages=[{'role': 'user', 'content': prompt_utente}],
            format={
                'type': 'object',
                'properties': {
                    'title': {'type': 'string'},
                    'body': {'type': 'string'}
                },
                'required': ['title', 'body']
            },
            options={
                'keep_alive': -1,  # -1 dice a Ollama: "Non scaricare MAI il modello finché non si chiude lo script"
                'temperature': 0.4
            }
        )
        dati_ticket = json.loads(response['message']['content'].strip())
        return dati_ticket['title'], dati_ticket['body']
    except Exception as e:
        print(f"Errore durante la generazione locale: {e}")
        return None, None


Semplicissimo ciclo for responsabile della creazione di *da_fare* ticket.

Componenti fondamentali:
- Definizione dei Pool di Input:
    - Delle semplicissime liste contenenti dei pool di categorie, priorità, livelli di esperienza, ecc... che verranno estratti casualmente all'interno del ciclo. Il loro compito è quello di fornire un incipit di partenza a Gemma, così da indirizzarla di volta in volta nella crezione del ticket (operazione necessaria dato che inizialmente il modello creava sempre ticket simili)
- L'interno del ciclo for
    - All'inizio è possibile notare la prensenza di vari random.choices pesati per creare una logica realistica dei dati che verranno realizzati
    - *titolo, corpo = genera_ticket_con_gemma(categoria, priorità, esperienza, colloqualità,incipit,titolo, attiva_typo)*, chiama la funzione precedentemente descritta e conserva all'interno delle variabili *titolo* e *corpo* il risultato
    - Viene creato un dizionario e convertito in un dataframe che verrà inserito in all'interno del file csv (in append o in write, in base al fatto se esista già o meno un record al suo interno)


In [82]:
categorie_disponibili = ["Tecnico", "Amministrazione", "Commerciale"]
priorita_disponibili = ["Bassa", "Media", "Alta"]
livelli_esperienza = [
    "Principiante", "Intermedio", "Esperto/Professionale", "Utente occasionale",
    "Primo utente", "Avanzato", "Primo utilizzo", "Utente che si crede esperto ma non lo è",
    "Tecnofobo / Anziano", "Nativo digitale", "Sviluppatore / Tecnico interno",     
    "Amministratore di sistema (Admin)","Utente pigro","Cliente VIP / Direzione",            
    "Utente metodico/Preciso","Ex-utente di un software concorrente"
]
livelli_colloquialita = [
    "Formale e distaccato", "Molto informale e colloquiale", "Frustrato/Arrabbiato",
    "Confuso", "Dettagliato", "Frettoloso", "Sarcastico", "Simpaticone", "Ansioso / Nel panico",
    "Disperato / Rassegnato", "Logorroico / Prolisso", "Telegrafico / Minimale","Passivo-aggressivo",
    "Gentile ed empatico", "Disorganizzato / Sconnesso", "Diffidente / Sospettoso"
]
pool_incipit_urgenti = [
    "Così non posso lavorare! ", 
    "Non so più che fare... ", 
    "Da stamattina sono completamente bloccato a ",
    "Ho un'urgenza assoluta: ",
    "Scrivo perché l'attività è interrotta: ",
    "Ho bisogno di risolvere subito ",
    "Siamo completamente fermi su ",
    "Blocco totale della produzione su ",
    "Disastro, non funziona più nulla dopo ",
    "Soluzione immediata richiesta per ",
    "È prioritario sbloccare ",
    "Impossibile procedere con il lavoro a causa di ",
    "Mi serve supporto ora, sono fermo su ",
    "Situazione critica su ",
    "Non si può andare avanti così! "
]

pool_incipit_normali = [
    "",
    "Ciao, ", 
    "Buongiorno. ", 
    "Buongiorno! ",
    "Eccomi di nuovo, ", 
    "Salve, ", 
    "Buongiorno, ", 
    "Scusate il disturbo, ", 
    "Ho un problema: ", 
    "Vi scrivo perché ", 
    "Creo questo ticket per ", 
    "Da stamattina sto provando a ", 
    "Dopo l'ultimo aggiornamento, ", 
    "Ho già provato a riavviare ", 
    "Potete dare un occhiata a ", 
    "Ho già provato a riavviare, ma ", 
    "Gentile supporto, vi contatto per ", 
    "Vorrei segnalare ", 
    "Ho necessità di segnalare ",
    "Avrei bisogno di un controllo su ",
    "Vi contatto per una anomalia riscontrata su ",
    "Riscontro un errore nel sistema quando provo a ",
    "Non riesco a completare la procedura di ",
    "Sembra esserci un bug su ",
    "Eccoci di nuovo... "
]

pool_incipit_bassa = [
    "",
    "Quando avete tempo, ",
    "Senza alcuna fretta, ",
    "Giusto per informazione, ",
    "Vi scrivo per una questione non urgente: ",
    "Avrei una curiosità su ",
    "Segnalo una piccolissima anomalia: ",
    "Un piccolo dubbio su ",
    "Nulla di bloccante, ma volevo segnalare che ",
    "Per il futuro sarebbe utile capire se ",
    "Solo una domanda di routine: ",
    "Quando capita, potreste dare un'occhiata a ",
    "Solo una richiesta di info su ",
    "Piccolo chiarimento per quando siete liberi: ",
    "Volevo solo lasciarvi un feedback riguardo a "
]

pool_titolo_urgenti = [
    "",
    "Urgente: non va ", 
    "Richiesta urgente: ", 
    "Assistenza immediata per ",
    "BLOCCO TOTALE: ",
    "CRITICO - Malfunzionamento su ",
    "EMERGENZA: ",
    "Interruzione servizio su "
]

pool_titolo_normali = [
    "",
    "Problema con ", 
    "Aiuto: ", 
    "Errore ", 
    "Bloccato su ", 
    "Info per ", 
    "Non va ", 
    "Chiarimento su ", 
    "Mancato funzionamento ", 
    "Procedura per ", 
    "Richiesta chiarimenti ",
    "Segnalazione errore su ",
    "Anomalia riscontrata in ",
    "Richiesta di supporto per "
]

pool_titolo_bassa = [
    "",
    "Richiesta info (non urgente): ",
    "Segnalazione minore su ",
    "Domanda generica: ",
    "Feedback su ",
    "Piccolo dubbio su ",
    "Suggerimento / Miglioria per ",
    "Info di routine per ",
    "Nota di aggiornamento su ",
    "Curiosità tecnica riguardo a "
]
reparti_aziendali = [
    "Reparto Logistica", "Ufficio Risorse Umane", "Ufficio Acquisti", 
    "Produzione", "Marketing", "Sede di Roma", "Filiale Estera", 
    "Reparto tecnico", "Cliente",
    "Ufficio Contabilità", "Controllo Qualità", "Ufficio Legale", 
    "Reparto Vendite", "Sede di Milano", "Assistenza Clienti", 
    "Ufficio Personale", "Management", "Direzione Generale", 
    "Fornitore Esterno"
]

strumenti_generici = [
    "gestionale interno", "portale dipendenti", "piattaforma cloud", 
    "software di rete", "sistema centrale", "CRM", "applicazione", 
    "servizio cloud"
    "database aziendale", "server centrale", "portale fornitori", 
    "applicazione di fatturazione", "pannello di controllo", 
    "sistema di login", "software di contabilità", "piattaforma HR", 
    "modulo richieste", "archivio digitale"
]

In [83]:
import pandas as pd
import random

percorso_csv = "output/gemma_v4.csv"

da_fare = 485

#controllo se è il primo inserimento o meno
try:
    df_esistente = pd.read_csv(percorso_csv)
    
    if not df_esistente.empty and 'id' in df_esistente.columns:
        ultimo_id_str = df_esistente['id'].iloc[-1]
        id_di_partenza = int(ultimo_id_str.split("-")[1]) + 1
    else:
        id_di_partenza = 0
        
    scrivi_header = False
    print(f"File CSV rilevato con Pandas. Riprendo dall'ID: TK-{id_di_partenza}")

except (FileNotFoundError, pd.errors.EmptyDataError):
    #parto da zero se il file non è stato trovato o è vuoto
    id_di_partenza = 0
    scrivi_header = True
    print("File CSV non trovato o vuoto. Creazione nuovo dataset da TK-0.")

print("-" * 50)

for i in range(0, da_fare):
    id_corrente_numerico = id_di_partenza + i
    stringa_id = f"TK-{id_corrente_numerico}"
    
    categoria = random.choice(categorie_disponibili)
    priorità = random.choice(priorita_disponibili)
    esperienza = random.choice(livelli_esperienza)
    colloqualità = random.choice(livelli_colloquialita)
    attiva_typo = random.choices([True, False], weights=[20, 80])[0]

    #si sceglie incipit in base alla priorità. si usa la meccanica dei pesi per creare rumore
    if priorità == "Alta":
        pool_scelta_titolo = random.choices([pool_titolo_urgenti, pool_titolo_normali], weights=[70, 30])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_urgenti, pool_incipit_normali], weights=[70, 30])[0]
        incipit = random.choice(pool_scelta_incipit)

    elif priorità == "Media":
        pool_scelta_titolo = random.choices([pool_titolo_urgenti, pool_titolo_normali, pool_titolo_bassa], weights=[20, 60, 20])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_urgenti, pool_incipit_normali, pool_incipit_bassa], weights=[20, 60, 20])[0]
        incipit = random.choice(pool_scelta_incipit)

    elif priorità == "Bassa":
        pool_scelta_titolo = random.choices([pool_titolo_normali, pool_titolo_bassa], weights=[30, 70])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_normali, pool_incipit_bassa], weights=[30, 70])[0]
        incipit = random.choice(pool_scelta_incipit)    

    disturbo = ""
    if random.randint(0,2)==1:
        if categoria == "Tecnico":
            disturbo = f"L'utente specifica di lavorare presso: {random.choice(reparti_aziendali)}"
        elif categoria == "Amministrazione":
            disturbo = f"L'operazione contabile avviene tramite lo strumento: {random.choice(strumenti_generici)}"
        else:  # Commerciale
            disturbo = f"L'utente menziona che l'offerta serve per il budget del: {random.choice(reparti_aziendali)} o per lo strumento  {random.choice(strumenti_generici)}"

    print(f"Generazione in corso... [{categoria} - {priorità} - {esperienza} - {colloqualità} - {attiva_typo}] ({i+1}/{da_fare})")
            
    titolo, corpo = genera_ticket_con_gemma(categoria, priorità, esperienza, colloqualità,incipit,titolo,disturbo, attiva_typo)
    
    if titolo and corpo:
        singolo_ticket = {
            "id": [stringa_id],
            "title": [titolo],
            "body": [corpo],
            "category": [categoria],
            "priority": [priorità]
        }
        df_singolo = pd.DataFrame(singolo_ticket)
        
        #se è il primo ticket assoluto creato da zero, scrive l'header e apre in 'w'
        if id_corrente_numerico == 0 and scrivi_header:
            df_singolo.to_csv(percorso_csv, index=False, mode='w', header=True)
            scrivi_header = False 
        else:
            df_singolo.to_csv(percorso_csv, index=False, mode='a', header=False)
            
        print(f"Ticket {stringa_id} Creato e SALVATO nel CSV!")
        print(f"Titolo: {titolo}")
        print(f"Corpo:  {corpo[:80]}...") 
        print("-" * 50)
        print()
    else:
        print("Generazione fallita. Salto al successivo.")
        print("-" * 50)

print(f"\nGenerazione conclusa con successo!")
print(f"Trovi tutti i dati aggiornati in tempo reale qui: '{percorso_csv}'.")


File CSV rilevato con Pandas. Riprendo dall'ID: TK-15
--------------------------------------------------
Generazione in corso... [Commerciale - Media - Sviluppatore / Tecnico interno - Dettagliato - False] (1/485)
Ticket TK-15 Creato e SALVATO nel CSV!
Titolo: Richiesta di supporto per negoziazione contratto commerciale
Corpo:  Ho bisogno di assistenza per finalizzare i termini della negoziazione commercial...
--------------------------------------------------

Generazione in corso... [Amministrazione - Alta - Utente metodico/Preciso - Diffidente / Sospettoso - False] (2/485)
Ticket TK-16 Creato e SALVATO nel CSV!
Titolo: Problema con gestione fatture pendenti urgente
Corpo:  Ho riscontrato un blocco nella generazione dei report contabili mensili. Le tran...
--------------------------------------------------

Generazione in corso... [Commerciale - Bassa - Utente pigro - Ansioso / Nel panico - False] (3/485)
Ticket TK-17 Creato e SALVATO nel CSV!
Titolo: Curiosità tecnica riguardo a lic

I ticket sono stati creati eseguendo il codice in modalità batch, variando ogni volta la struttura del prompt, inserendo o rimuovendo dettagli, con lo scopo di rendere il dataset più eterogeneo. I prompt e gli incipit presenti in questo Notebook sono quelli che hanno dato dei risultati più bilanciati. 

Ho deciso per tal motivo di effettuare un piccolo shuffling dei record, mantenendo ovviamente l'ordine degli id, con lo scopo di neutralizzare eventuali bias legati alla sequenzialità delle varie fasi. 

In [84]:
import pandas as pd
percorso_csv = "output/ticket_utenti_gemma_shuffled.csv"
df = pd.read_csv("output/gemma_v2.csv")
#isolo per conservare la colonna degli id
ids = df['id'].copy()
#mischio casualmente tutte le atre colonne
df_features_shuffled = df[['title', 'body', 'category', 'priority']].sample(frac=1, random_state=42).reset_index(drop=True)
df_omogeneo = pd.concat([ids, df_features_shuffled], axis=1)
df_omogeneo.to_csv(percorso_csv, index=False, mode='w', header=True)

# Conclusioni

- Inizialmente la scelta era ricaduta sulla versione standard e completa del modello Gemma 4. Sebbene l'accuratezza e la diversificazione dei testi fossero ottimali, ho notato una complessità computazionale non da ignorare ed sono andato alla ricerca di alternative più leggere e sostenibili localmente.

- Ho pertanto sperimentato l'introduzione di varianti ottimizzate e quantizzate a parametri ridotti, nello specifico le release da 2 miliardi (e2b) e 4 miliardi di parametri (e4b). Sebbene queste architetture abbiano garantito un'esecuzione radicalmente più fulminea e un'impronta di memoria minima sul sistema, i test sul campo hanno evidenziato una forte "pigrizia semantica"; si sono presentati fenomeni di ripetitività strutturale, generando schemi sintattici quasi identici per i ticket appartenenti alle medesime categorie (quasi nulla di diverso dai ticket generati nel Jupyter 01).

- Poiché l'eterogeneità e la variabilità semantica del dataset sono requisiti fondamentali per evitare l'overfitting e garantire una corretta generalizzazione dei successivi classificatori di Machine Learning, ho stabilito che la qualità del dato dovesse prevalere sulle tempistiche di computazione. Sono dunque ritornato definitivamente al modello base standard di Gemma 4, accettando requisiti hardware più stringenti in cambio di un dataset testuale autentico, eterogeneo e strutturalmente solido.

## Possibili Miglioramenti

Questo script mostra diversi possibili miglioramenti:
- Diversa strutturazione dei prompt: è possibile utilizzare diversi approcci di prompt engineering per avere risultati diversi e adatti alla situazione
- Diversi modelli di intelligenza artificiale: ho deciso di utilizzare Gemma4 perché la conoscevo già, però ovviamente non è l'unico modello di intelligenza artificiale con la quale si può sfruttare questa pipeline, basta dare un occhiata alla sezione model del sito web ufficiale di [ollama](https://ollama.com/search) per poterne visualizzare qualcuno.

Diamo un occhiata ai ticket generati da Gemma per validarne la qualità prima di procedere con la generazione del numero di ticket richiesto dal PW.

In [85]:
df_anteprima = pd.read_csv("output/ticket_utenti_gemma.csv")
print("*** TICKET GEMMA ***")
df_anteprima.head(5)["body"]

*** TICKET GEMMA ***


0    Ciao, vorrei richiedere un chiarimento relativ...
1    Vi scrivo perché ho un dubbio sui prezzi che m...
2    Vi scrivo perché non riesco ad accedere alla s...
3    Buongiorno, ho notato che alcuni articoli nel ...
4    Salve, scusa ma non riesco ad accedere al port...
Name: body, dtype: str

__Una EDA più accurata è disponibile all'interno del Jupyter 03.__